# Tutorial 13: Transformer Fine-Tuning
## Download and adapt a pretrained vision transformer for a specific object-detection task

**Course:** IE 1171  
**Example dataset downloaded:** `keremberke/license-plate-object-detection`  
**Pretrained model downloaded:** `hustvl/yolos-tiny`  
**Level 1:** Required core—data audit, bounding boxes, transfer learning, fine-tuning, evaluation, and a prediction pipeline  
**Level 2:** Optional deep dive—longer training, outside-data testing, and safer video use

---

Claude and the model in this tutorial are different tools. Claude is a large, general-purpose language model that is accessed through a chat interface. In this notebook, you download a smaller task-specific transformer and change its weights with labeled images. Claude can help explain and coordinate the code, but it does not perform the training for you.

The object-detection example finds regions that should be blurred for privacy. The main subject of the tutorial is **transformers, transfer learning, and fine-tuning**. The student must still verify the data source, annotation format, training split, evaluation threshold, missed detections, and final claim.


# <img src="tutorial-icons/learning_objectives.png" alt="Learning Objectives" width="44" style="vertical-align:middle; margin-right:10px;"> Learning Objectives

By the end of this tutorial, you should be able to:

1. **Compare Claude with a task-specific transformer**
   - Explain that Claude is a general-purpose language model, while this notebook fine-tunes a smaller vision model for one output.
   - Identify shared ideas such as tokens, embeddings, attention, pretrained weights, and fine-tuning.
   - Explain why a model built for text generation is different from a model built to return object locations.
2. **Fine-tune a downloaded transformer**
   - Interpret image annotations, bounding boxes, intersection over union, and the model's detection loss.
   - Replace the pretrained output head and fine-tune `hustvl/yolos-tiny` on labeled examples.
   - Keep training, validation, and test data in the correct roles.
3. **Evaluate and use the model carefully**
   - Compare precision, recall, $F_2$, average precision, and confidence thresholds.
   - Turn predicted boxes into padded blur regions for the example task.
   - Explain why a model output still needs human review before public use.


# Pólya’s Four-Step Problem-Solving Cycle

> **Backbone for this tutorial:** George Pólya’s four steps organize the work from problem framing through verification. The steps are a **cycle**, not a one-way checklist: if later evidence exposes a bad assumption, return to the earlier step that needs revision.

| Marker | Pólya step | Guiding question | In this tutorial |
|---|---|---|---|
| **🔵 🧭** | **Understand the Problem** | What is the real problem, what is known, and what constraints define success? | Define the plate-redaction goal, privacy success criteria, dataset boundaries, and failure consequences. |
| **🟣 🗺️** | **Devise a Plan** | What sequence of actions and checks should connect the current state to the goal? | Plan the data audit, model adaptation, threshold selection, and evaluation strategy before fine-tuning. |
| **🟠 🛠️** | **Carry Out the Plan** | Can the plan be executed in small, observable steps and checked as it runs? | Fine-tune and apply the detector/redaction pipeline while preserving validation and test separation. |
| **🟢 🔎** | **Look Back** | Does the result answer the original problem, and what should be revised or generalized? | Compare validation, test, and external failures and reflect on privacy tradeoffs and limitations. |

The colored markers reappear at the points where each step becomes the main focus. **Human Checks support the cycle, but they are not a universal checklist:** meaningful verification depends on domain knowledge, the data-generating process, and the consequences of being wrong.


# <img src="tutorial-icons/assigned_reading.png" alt="Assigned Reading" width="44" style="vertical-align:middle; margin-right:10px;"> Assigned Reading

Read the following before or alongside the notebook:

- Hugging Face, [Object Detection Task Guide](https://huggingface.co/docs/transformers/tasks/object_detection).
- Hugging Face, [YOLOS documentation](https://huggingface.co/docs/transformers/model_doc/yolos).
- HUST Vision Lab, [`hustvl/yolos-tiny` model card](https://huggingface.co/hustvl/yolos-tiny).
- Keremberke, [`license-plate-object-detection` dataset card](https://huggingface.co/datasets/keremberke/license-plate-object-detection).
- Fang et al., [You Only Look at One Sequence](https://arxiv.org/abs/2106.00666).

Focus on transfer learning, detection tokens, bounding-box losses, Hungarian matching, object-detection metrics, dataset licensing, and the difference between model output and a reliable privacy guarantee.

## 🔵 🧭 Pólya Step 1 — Understand the Problem

**Backbone checkpoint.** State the real goal, evidence, constraints, and what would count as success before asking an AI system to solve anything.

**In this tutorial:** Define the plate-redaction goal, privacy success criteria, dataset boundaries, and failure consequences.

# Example Task and Dataset

The example uses **`keremberke/license-plate-object-detection`**.

| Selection factor | Evidence | Why it fits this tutorial |
|---|---|---|
| Size | 8,823 images | Large enough for meaningful fine-tuning without being enormous. |
| Download | About 230 MB | Practical for a Colab session. |
| Splits | 6,176 train; 1,765 validation; 882 test | No need to invent a split after inspecting all images. |
| Annotation | COCO `[x_min, y_min, width, height]` boxes | Directly supports object-detection training. |
| Classes | One class: `license_plate` | Keeps the learning objective focused. |
| License | CC BY 4.0 | Reuse is allowed with attribution. |
| Access | `datasets.load_dataset(...)` | No Kaggle credential or hosted inference API is required. |

## Alternatives considered

| Dataset | Strength | Reason it is not the Level 1 default |
|---|---|---|
| PP4AV | 3,447 realistic European driving images; designed for privacy evaluation | About 4.51 GB and CC BY-NC-ND 4.0; better as an external benchmark than a simple training download. |
| Roboflow License Plate Recognition | 10,125 images; CC BY 4.0 | Good alternative, but direct Hugging Face loading makes the selected dataset easier for one notebook. |
| Kaggle Car Plate Detection | Small and simple | Only 433 images and requires the Kaggle download workflow. |
| Open Images | Very diverse and professionally annotated | Much larger and requires class-specific download and conversion work. |

> **Scope:** The tutorial detects the rectangular plate region and blurs it. It does **not** read, store, or recognize the plate characters.

# <img src="tutorial-icons/tutorial_flow.png" alt="Tutorial Flow" width="44" style="vertical-align:middle; margin-right:10px;"> Tutorial Flow

| Part | Purpose |
|---|---|
| **1. Environment** | Confirm GPU, install libraries, and record versions. |
| **2. Data** | Download, audit, and visualize licensed plate annotations. |
| **3. Model** | Download YOLOS and replace its COCO classification head. |
| **4. Representation** | Convert images and boxes into model inputs and labels. |
| **5. Fine-tuning** | Adapt pretrained weights on training rows only. |
| **6. Evaluation** | Select a threshold on validation data; evaluate test once. |
| **7. Redaction** | Pad and blur predicted plate boxes. |
| **8. Level 2** | Train more fully and stress-test distribution shift and video behavior. |

## Tutorial Symbols

| Symbol | Meaning | What to do |
|---|---|---|
| **🔵 🧭  🟣 🗺️  🟠 🛠️  🟢 🔎** | **Pólya Backbone** | Treat the four colored checkpoints as the main problem-solving cycle; return to an earlier step when new evidence requires revision. |
| <img src="tutorial-icons/tutorial_flow.png" alt="Tutorial Flow" width="28" style="vertical-align:middle; margin-right:8px;"> | **Tutorial Flow** | Follow the notebook's normal route. |
| <img src="tutorial-icons/learning_objectives.png" alt="Learning Objectives" width="28" style="vertical-align:middle; margin-right:8px;"> | **Learning Objectives** | See the three destinations for the tutorial. |
| <img src="tutorial-icons/assigned_reading.png" alt="Assigned Reading" width="28" style="vertical-align:middle; margin-right:8px;"> | **Assigned Reading** | Read the named sections before or alongside the notebook. |
| <img src="tutorial-icons/theory.png" alt="Theory" width="28" style="vertical-align:middle; margin-right:8px;"> | **Theory** | Connect equations, assumptions, and concepts to the current part. |
| <img src="tutorial-icons/manual_pause.png" alt="Manual Pause" width="28" style="vertical-align:middle; margin-right:8px;"> | **Manual Pause** | Think or predict before asking Claude. |
| <img src="tutorial-icons/without_claude.png" alt="Without Claude" width="28" style="vertical-align:middle; margin-right:8px;"> | **Without Claude** | Notice the details an AI collaborator can coordinate. |
| <img src="tutorial-icons/claude_task.png" alt="Claude Task" width="28" style="vertical-align:middle; margin-right:8px;"> | **Claude Task** | Use one focused and checkable prompt. |
| <img src="tutorial-icons/your_workspace.png" alt="Your Workspace" width="28" style="vertical-align:middle; margin-right:8px;"> | **Your Workspace** | Paste, read, and run the response. |
| <img src="tutorial-icons/reference_solution.png" alt="Reference Solution" width="28" style="vertical-align:middle; margin-right:8px;"> | **Reference Solution** | Compare only after your own attempt. |
| <img src="tutorial-icons/human_check.png" alt="Human Check" width="28" style="vertical-align:middle; margin-right:8px;"> | **Human Check — domain expertise required** | Use the provided questions, then add a domain-specific check. The notebook cannot supply a complete checklist for every application. |
| <img src="tutorial-icons/look_back.png" alt="Look Back" width="28" style="vertical-align:middle; margin-right:8px;"> | **Look Back** | Interpret, challenge, and reflect on the result. |
| <img src="tutorial-icons/level_2.png" alt="Level 2 Challenge" width="28" style="vertical-align:middle; margin-right:8px;"> | **Level 2 — Challenge Ahead** | Take the optional harder route after Level 1. |

> **Important — Human Checks are not a complete checklist.** The notebook can suggest generic verification questions, but deciding what *must* be checked depends on knowledge of the domain, the data-generating process, and the consequences of an error. If you do not have that expertise, involve someone who does. Every Human Check asks you to add your own domain-specific question.

# <img src="tutorial-icons/theory.png" alt="Theory" width="44" style="vertical-align:middle; margin-right:10px;"> Theory Foundation: Transformers, Inputs, Outputs, and Fine-Tuning

A **transformer** is a model design that uses attention. Claude is one example: it is a very large transformer trained mainly to work with language. The model downloaded in this notebook is a smaller vision transformer trained to work with image patches and object locations.

The two models share ideas such as tokens, embeddings, attention, pretrained weights, and fine-tuning. They are still different because they receive different inputs, return different outputs, use different losses, and are tested with different measures.

## Why Claude is not the detector used here

Blurring requires pixel coordinates $(x_{min},y_{min},x_{max},y_{max})$. A text-only LLM has no visual input and cannot calculate those coordinates from image pixels. Even a vision-language model that can discuss an image may produce imprecise boxes. This tutorial therefore downloads a **vision transformer for object detection**. The conceptual bridge to LLMs is real—tokens, embeddings, attention, pretraining, and fine-tuning—but the task-specific model must match the output required.

## Four different vision tasks

| Task | Output | Plate example |
|---|---|---|
| Image classification | one label for the whole image | “This image contains a vehicle.” |
| Object detection | class plus bounding box for each object | “A plate is at these coordinates.” |
| OCR | character sequence | reading the plate number |
| Redaction | modified pixels | blurring the detected region |

We need detection before redaction. OCR is deliberately excluded because recognizing the characters would collect more sensitive information than is necessary to hide them.

## YOLOS representation

For patch size $P$, an image with height $H$ and width $W$ becomes approximately

$$
N=\frac{HW}{P^2}
$$

patch tokens. After projection and position embeddings, self-attention updates token representations:

$$
\operatorname{Attention}(Q,K,V)=
\operatorname{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V.
$$

YOLOS adds learned **detection tokens**. Each detection token predicts a class distribution and a normalized box. The model uses a fixed set of queries, so most predictions become “no object” and the remaining predictions represent plates.

## Fine-tuning

Let $\theta_0$ be weights learned from ImageNet and COCO. Fine-tuning starts at $\theta_0$ rather than a random point and minimizes a plate-specific loss:

$$
\theta^*=\arg\min_{\theta}\frac{1}{n}\sum_{i=1}^{n}
L\bigl(f_\theta(x_i),y_i\bigr).
$$

The old COCO head predicts many classes and must be replaced with a one-class plate head plus “no object.” `ignore_mismatched_sizes=True` permits this replacement while retaining compatible pretrained weights.

## Privacy objective

For redaction, a false negative leaves a plate visible; a false positive blurs extra background. These harms are not symmetric. A validation threshold should generally favor recall, but lowering it too far can blur large amounts of irrelevant content. The threshold is therefore a decision policy attached to the model, not a property learned automatically.

### Questions you should be ready to answer

- Which transformer concepts are shared between language and vision?
- Why is object detection required before blurring?
- What changes when the pretrained class head is replaced?
- Why should validation—not test—choose the confidence threshold?
- Why is high average precision still not a privacy guarantee?

# Level 1 — Required Core

# Part 1: Prepare a Reproducible GPU Environment

A CPU can load the dataset, inspect annotations, and run small tests, but fine-tuning performs many large matrix operations and is much slower on a CPU. A GPU can carry out these operations in parallel. **The training steps in this tutorial require a CUDA-capable GPU.** In Colab, choose **Runtime → Change runtime type → T4 GPU** before running the installation cell. Record package versions because model APIs and defaults can change.

## <img src="tutorial-icons/claude_task.png" alt="Claude Task" width="36" style="vertical-align:middle; margin-right:9px;"> Claude Coding Task 1: Install and Audit the Environment

```text
Write two Jupyter cells.

Cell 1 must install or update datasets, transformers, accelerate, torchmetrics,
pycocotools, matplotlib, and Pillow.

Cell 2 must:
1. import torch, transformers, datasets, PIL, and platform;
2. set random seeds to 1099;
3. print Python, PyTorch, Transformers, and Datasets versions;
4. print whether CUDA is available and the GPU name;
5. raise a clear RuntimeError if CUDA is unavailable;
6. create device = torch.device("cuda").

Return only the two code cells.
```

### <img src="tutorial-icons/your_workspace.png" alt="Your Workspace" width="30" style="vertical-align:middle; margin-right:8px;"> Your Workspace

Paste, inspect, and run Claude's two cells below.

### <img src="tutorial-icons/reference_solution.png" alt="Reference Solution" width="30" style="vertical-align:middle; margin-right:8px;"> Reference Solution for Task 1

In [ ]:
!pip -q install -U datasets transformers accelerate torchmetrics pycocotools matplotlib Pillow

In [ ]:
import platform
import random
import numpy as np
import torch
import transformers
import datasets
import PIL

SEED = 1099
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Enable a GPU runtime before fine-tuning.")
print("GPU:", torch.cuda.get_device_name(0))
device = torch.device("cuda")

# Part 2: Download and Audit the Dataset

The dataset card reports COCO-format boxes:

$$
b=[x_{min},y_{min},w,h].
$$

The bottom-right corner is $(x_{min}+w,y_{min}+h)$. Valid boxes require $w>0$, $h>0$, $x_{min}\ge0$, $y_{min}\ge0$, and coordinates within the image. These checks should occur before training because corrupted geometry can create invalid losses.

Level 1 uses a reproducible subset—1,500 training images, 300 validation images, and 300 test images—to keep the notebook practical. The original split identities remain unchanged. Level 2 removes the limits.

## <img src="tutorial-icons/claude_task.png" alt="Claude Task" width="36" style="vertical-align:middle; margin-right:9px;"> Claude Coding Task 2: Load and Validate Dataset Structure

```text
Using datasets.load_dataset:

1. load keremberke/license-plate-object-detection with name="full";
2. print the dataset object, split sizes, features, license reminder, and one
   example's keys without printing or reading plate characters;
3. preserve the supplied train, validation, and test splits;
4. shuffle each split with seed 1099 and select at most 1500 train,
   300 validation, and 300 test images;
5. name them train_raw, validation_raw, and test_raw;
6. count images, boxes, zero-box images, and invalid COCO boxes in every subset;
7. assert that all categories equal 0 and every box is within image bounds;
8. fit no model.

Return only the Python code.
```

### <img src="tutorial-icons/your_workspace.png" alt="Your Workspace" width="30" style="vertical-align:middle; margin-right:8px;"> Your Workspace

Paste, read, and run Claude's response in the next cell.

### <img src="tutorial-icons/reference_solution.png" alt="Reference Solution" width="30" style="vertical-align:middle; margin-right:8px;"> Reference Solution for Task 2

In [ ]:
from datasets import load_dataset

plate_data = load_dataset(
    "keremberke/license-plate-object-detection",
    name="full",
)
print(plate_data)
print("Features:", plate_data["train"].features)
print("License reminder: CC BY 4.0; preserve attribution.")
print("Example keys:", plate_data["train"][0].keys())

limits = {"train": 1500, "validation": 300, "test": 300}
subsets = {}
for split_name, limit in limits.items():
    shuffled = plate_data[split_name].shuffle(seed=SEED)
    subsets[split_name] = shuffled.select(range(min(limit, len(shuffled))))

train_raw = subsets["train"]
validation_raw = subsets["validation"]
test_raw = subsets["test"]

def audit_split(split):
    boxes = zero_box = invalid = 0
    for row in split:
        row_boxes = row["objects"]["bbox"]
        row_categories = row["objects"]["category"]
        boxes += len(row_boxes)
        zero_box += int(len(row_boxes) == 0)
        assert all(int(category) == 0 for category in row_categories)
        for x, y, width, height in row_boxes:
            good = (
                x >= 0 and y >= 0 and width > 0 and height > 0
                and x + width <= row["width"] + 1e-4
                and y + height <= row["height"] + 1e-4
            )
            invalid += int(not good)
    return {"images": len(split), "boxes": boxes,
            "zero_box_images": zero_box, "invalid_boxes": invalid}

for name, split in subsets.items():
    summary = audit_split(split)
    print(name, summary)
    assert summary["invalid_boxes"] == 0

## <img src="tutorial-icons/human_check.png" alt="Human Check" width="30" style="vertical-align:middle; margin-right:8px;"> Human Check

> **Domain expertise required:** The questions below are examples, not a complete checklist. Add a check based on the real application; if you lack that expertise, involve someone who has it.

- **Your own domain question:** What could be wrong here that a generic AI system or checklist would be unlikely to notice?


- Did the code use the supplied split names rather than create a random split from all rows?
- Are category IDs consistently zero?
- Are any images missing boxes?
- Do all boxes remain inside image boundaries?
- Is the Level 1 subset selected reproducibly?
- Was the dataset license recorded?
- Did any step attempt to recognize or store plate text?

# Part 3: Visualize the Ground-Truth Geometry

Never fine-tune an object detector without viewing annotations. A structurally valid box can still cover the wrong object, be too loose, or omit a visible plate. Annotation inspection tests the meaning of the labels, not only their numeric range.

## <img src="tutorial-icons/theory.png" alt="Theory" width="36" style="vertical-align:middle; margin-right:9px;"> Theory: Intersection Over Union Measures Localization Overlap

For predicted box $B_p$ and ground-truth box $B_g$,

$$
IoU(B_p,B_g)=\frac{|B_p\cap B_g|}{|B_p\cup B_g|}.
$$

$IoU=1$ means perfect overlap and $IoU=0$ means no overlap. A threshold such as 0.50 decides whether a prediction localizes a ground-truth object well enough to count as a match. In privacy redaction, a slightly loose box is often safer than a tight box that exposes plate edges. Therefore the blur function later adds padding even after detection.

## <img src="tutorial-icons/claude_task.png" alt="Claude Task" width="36" style="vertical-align:middle; margin-right:9px;"> Claude Coding Task 3: Draw and Inspect Ground-Truth Boxes

```text
Using train_raw:

1. select six reproducible examples;
2. draw every COCO bounding box in orange with a navy outline;
3. display the images in a 2-by-3 grid with image dimensions and box counts;
4. create a dataframe of relative box area = box_area / image_area;
5. print quantiles at 0, .10, .25, .50, .75, .90, and 1;
6. do not display or extract plate characters separately.

Return only the Python code.
```

### <img src="tutorial-icons/your_workspace.png" alt="Your Workspace" width="30" style="vertical-align:middle; margin-right:8px;"> Your Workspace

Paste, read, and run Claude's response in the next cell.

### <img src="tutorial-icons/reference_solution.png" alt="Reference Solution" width="30" style="vertical-align:middle; margin-right:8px;"> Reference Solution for Task 3

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for ax, index in zip(axes.ravel(), [0, 7, 19, 41, 83, 127]):
    row = train_raw[index]
    image = row["image"].convert("RGB")
    ax.imshow(image)
    for x, y, width, height in row["objects"]["bbox"]:
        ax.add_patch(Rectangle((x, y), width, height, fill=False,
                               edgecolor="#F4A01C", linewidth=3))
    ax.set_title(f'{row["width"]}×{row["height"]}; '
                 f'{len(row["objects"]["bbox"])} box(es)')
    ax.axis("off")
plt.tight_layout()
plt.show()

relative_areas = []
for row in train_raw:
    image_area = row["width"] * row["height"]
    for area in row["objects"]["area"]:
        relative_areas.append(area / image_area)
display(pd.Series(relative_areas, name="relative_box_area")
        .quantile([0, .10, .25, .50, .75, .90, 1]).to_frame())

## 🟣 🗺️ Pólya Step 2 — Devise a Plan

**Backbone checkpoint.** Decide the sequence of actions and checks before the main execution. Make assumptions, evaluation rules, and stopping conditions visible so they can be challenged.

**In this tutorial:** Plan the data audit, model adaptation, threshold selection, and evaluation strategy before fine-tuning.

# Part 4: Download YOLOS and Replace the Detection Head

`hustvl/yolos-tiny` has approximately 6.49 million parameters. It was pretrained on ImageNet-1k and fine-tuned on COCO object detection. We transfer its patch representation and attention weights, then replace its many-class COCO head with the one-class plate head.

The checkpoint uses the Apache 2.0 model license. The dataset uses CC BY 4.0. Model and data licenses are separate and both must be recorded.

## <img src="tutorial-icons/theory.png" alt="Theory" width="36" style="vertical-align:middle; margin-right:9px;"> Theory: Transfer Learning Changes the Starting Point

Training from scratch initializes $\theta$ randomly. Transfer learning initializes at $\theta_0$, where patch and attention weights already represent useful visual structure. With limited plate data, this often reduces training time and sample requirements.

Fine-tuning every parameter can adapt strongly but may overwrite useful representations. Freezing the backbone reduces computation but restricts adaptation. Level 1 fine-tunes the full small model with a low learning rate. That is a design choice to validate, not a universal rule.

The number of trainable parameters should be printed. “Tiny” is relative: millions of parameters can still memorize patterns, require a GPU, and fail under distribution shift.

## <img src="tutorial-icons/claude_task.png" alt="Claude Task" width="36" style="vertical-align:middle; margin-right:9px;"> Claude Coding Task 4: Load the Processor and Plate Detector

```text
Using MODEL_NAME = "hustvl/yolos-tiny":

1. load AutoImageProcessor at a fixed 480-by-480 training size;
2. create id2label and label2id for one class named license_plate;
3. load AutoModelForObjectDetection with the mappings and
   ignore_mismatched_sizes=True;
4. print the model type, total parameters, trainable parameters,
   num_labels, and label maps;
5. explain through comments why the classification-head warning is expected;
6. do not train yet.

Return only the Python code.
```

### <img src="tutorial-icons/your_workspace.png" alt="Your Workspace" width="30" style="vertical-align:middle; margin-right:8px;"> Your Workspace

Paste, read, and run Claude's response in the next cell.

### <img src="tutorial-icons/reference_solution.png" alt="Reference Solution" width="30" style="vertical-align:middle; margin-right:8px;"> Reference Solution for Task 4

In [ ]:
from transformers import AutoImageProcessor, AutoModelForObjectDetection

MODEL_NAME = "hustvl/yolos-tiny"
id2label = {0: "license_plate"}
label2id = {"license_plate": 0}

image_processor = AutoImageProcessor.from_pretrained(
    MODEL_NAME,
    size={"height": 480, "width": 480},
)
model = AutoModelForObjectDetection.from_pretrained(
    MODEL_NAME,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,  # expected: replace the COCO class head
)

total_parameters = sum(parameter.numel() for parameter in model.parameters())
trainable_parameters = sum(
    parameter.numel() for parameter in model.parameters()
    if parameter.requires_grad
)
print("Model type:", model.config.model_type)
print("Total parameters:", f"{total_parameters:,}")
print("Trainable parameters:", f"{trainable_parameters:,}")
print("Number of labels:", model.config.num_labels)
print("id2label:", model.config.id2label)
print("label2id:", model.config.label2id)

# Part 5: Convert Dataset Rows Into Model Inputs

The image processor rescales pixels and converts COCO annotations into the normalized target format expected by YOLOS. Training examples contain `pixel_values` and a label dictionary with class labels and boxes.

Because Level 1 uses a fixed 480-by-480 processor size, tensors can be stacked into batches. The original image size is still required later to convert normalized predictions back to pixel coordinates.

## <img src="tutorial-icons/theory.png" alt="Theory" width="36" style="vertical-align:middle; margin-right:9px;"> Theory: Detection Loss Must Solve Classification, Localization, and Matching

YOLOS predicts a fixed set of queries. Hungarian matching assigns predictions to ground-truth objects one-to-one by minimizing a cost. A simplified loss is

$$
L=\lambda_{cls}L_{CE}+\lambda_1\lVert b-\hat b\rVert_1
+\lambda_{giou}L_{GIoU}.
$$

Cross-entropy distinguishes `license_plate` from “no object.” $L_1$ penalizes coordinate differences. Generalized IoU penalizes poor overlap even when boxes do not intersect. The training loss combines these objectives; a smaller total loss does not tell us which privacy errors remain, so images and detection metrics must also be inspected.

## <img src="tutorial-icons/claude_task.png" alt="Claude Task" width="36" style="vertical-align:middle; margin-right:9px;"> Claude Coding Task 5: Build a PyTorch Dataset and Collator

```text
Using train_raw, validation_raw, test_raw, and image_processor:

1. define LicensePlateDataset as a torch Dataset;
2. convert every row's objects into COCO annotation dictionaries containing
   id, image_id, category_id, bbox, area, and iscrowd=0;
3. call image_processor with the RGB image and annotations;
4. return pixel_values without its batch dimension and labels for one image;
5. create train_dataset, validation_dataset, and test_dataset;
6. define a collate_fn that stacks pixel_values and keeps labels as a list;
7. inspect one item and one two-image DataLoader batch;
8. assert pixel dimensions are 480 by 480.

Return only the Python code.
```

### <img src="tutorial-icons/your_workspace.png" alt="Your Workspace" width="30" style="vertical-align:middle; margin-right:8px;"> Your Workspace

Paste, read, and run Claude's response in the next cell.

### <img src="tutorial-icons/reference_solution.png" alt="Reference Solution" width="30" style="vertical-align:middle; margin-right:8px;"> Reference Solution for Task 5

In [ ]:
from torch.utils.data import Dataset, DataLoader

class LicensePlateDataset(Dataset):
    def __init__(self, split, processor):
        self.split = split
        self.processor = processor

    def __len__(self):
        return len(self.split)

    def __getitem__(self, index):
        row = self.split[index]
        annotations = []
        for object_id, area, bbox, category in zip(
            row["objects"]["id"],
            row["objects"]["area"],
            row["objects"]["bbox"],
            row["objects"]["category"],
        ):
            annotations.append({
                "id": int(object_id),
                "image_id": int(row["image_id"]),
                "category_id": int(category),
                "bbox": [float(value) for value in bbox],
                "area": float(area),
                "iscrowd": 0,
            })
        target = {"image_id": int(row["image_id"]),
                  "annotations": annotations}
        encoded = self.processor(
            images=row["image"].convert("RGB"),
            annotations=target,
            return_tensors="pt",
        )
        return {
            "pixel_values": encoded["pixel_values"].squeeze(0),
            "labels": encoded["labels"][0],
        }

def collate_fn(batch):
    return {
        "pixel_values": torch.stack([item["pixel_values"] for item in batch]),
        "labels": [item["labels"] for item in batch],
    }

train_dataset = LicensePlateDataset(train_raw, image_processor)
validation_dataset = LicensePlateDataset(validation_raw, image_processor)
test_dataset = LicensePlateDataset(test_raw, image_processor)

sample = train_dataset[0]
print(sample["pixel_values"].shape, sample["labels"].keys())
assert sample["pixel_values"].shape[-2:] == (480, 480)
batch = next(iter(DataLoader(train_dataset, batch_size=2,
                             collate_fn=collate_fn)))
print("Batch pixels:", batch["pixel_values"].shape)
print("Label dictionaries:", len(batch["labels"]))

## 🟠 🛠️ Pólya Step 3 — Carry Out the Plan

**Backbone checkpoint.** Execute in small, observable steps. Read generated code or actions, stay within scope, and compare outputs with the behavior you predicted.

**In this tutorial:** Fine-tune and apply the detector/redaction pipeline while preserving validation and test separation.

# Part 6: Fine-Tune on Training Images Only

The required run uses one epoch so it can finish in a normal Colab session. This is a teaching run, not a production model. The validation split reports selection evidence; the test split remains untouched.

The effective batch size is

$$
B_{effective}=B_{device}\times G,
$$

where $G$ is `gradient_accumulation_steps`. Accumulation allows a larger effective batch when GPU memory is limited.

## <img src="tutorial-icons/manual_pause.png" alt="Manual Pause" width="36" style="vertical-align:middle; margin-right:9px;"> Manual Pause: Predict Before Training

1. Which layers are newly initialized?
2. Why use a smaller learning rate than training from scratch?
3. What evidence would suggest overfitting?
4. Why is the test split excluded from threshold selection?
5. If the GPU runs out of memory, which batch setting should change first?
6. Why does one epoch make this an instructional baseline rather than a finished anonymizer?

## <img src="tutorial-icons/claude_task.png" alt="Claude Task" width="36" style="vertical-align:middle; margin-right:9px;"> Claude Coding Task 6: Configure and Run Fine-Tuning

```text
Using transformers.Trainer:

1. create TrainingArguments with output_dir="plate-yolos-level1";
2. use one epoch, learning_rate=5e-5, weight_decay=1e-4,
   per_device_train_batch_size=4, per_device_eval_batch_size=4,
   gradient_accumulation_steps=2, warmup_ratio=.10, fp16=True,
   eval_strategy="epoch", save_strategy="epoch", logging_steps=25,
   remove_unused_columns=False, report_to="none", save_total_limit=1,
   load_best_model_at_end=True, and seed=1099;
3. pass model, train_dataset, validation_dataset, image_processor,
   and collate_fn to Trainer;
4. print the effective batch size;
5. call train and save_model("plate-yolos-level1/final");
6. name the final fitted model plate_model;
7. do not use test_dataset.

Return only the Python code.
```

### <img src="tutorial-icons/your_workspace.png" alt="Your Workspace" width="30" style="vertical-align:middle; margin-right:8px;"> Your Workspace

Paste, read, and run Claude's response in the next cell.

### <img src="tutorial-icons/reference_solution.png" alt="Reference Solution" width="30" style="vertical-align:middle; margin-right:8px;"> Reference Solution for Task 6

In [ ]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="plate-yolos-level1",
    num_train_epochs=1,
    learning_rate=5e-5,
    weight_decay=1e-4,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,
    warmup_ratio=0.10,
    fp16=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=25,
    remove_unused_columns=False,
    report_to="none",
    save_total_limit=1,
    load_best_model_at_end=True,
    seed=SEED,
)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    processing_class=image_processor,
    data_collator=collate_fn,
)
print("Effective batch size:",
      training_args.per_device_train_batch_size
      * training_args.gradient_accumulation_steps)
trainer.train()
trainer.save_model("plate-yolos-level1/final")
plate_model = trainer.model

## <img src="tutorial-icons/human_check.png" alt="Human Check" width="30" style="vertical-align:middle; margin-right:8px;"> Human Check

> **Domain expertise required:** The questions below are examples, not a complete checklist. Add a check based on the real application; if you lack that expertise, involve someone who has it.

- **Your own domain question:** What could be wrong here that a generic AI system or checklist would be unlikely to notice?


- Did training use only `train_dataset`?
- Did validation occur only at the planned epoch boundary?
- Is the test dataset still untouched?
- Did loss remain finite?
- Was the final model saved locally?
- How many training steps were completed?
- Why is lower training loss not enough to claim reliable plate redaction?

# Part 7: Select a Privacy-Oriented Threshold

A detector produces many boxes with confidence scores. The threshold $t$ controls which boxes become redactions.

At IoU 0.50:

$$
Precision=\frac{TP}{TP+FP},\qquad Recall=\frac{TP}{TP+FN}.
$$

For privacy, recall is especially important because $FN$ leaves a real plate visible. The $F_2$ score weights recall more heavily:

$$
F_2=\frac{5\,Precision\,Recall}{4\,Precision+Recall}.
$$

Average precision summarizes the precision–recall curve across confidence thresholds. AP50 requires localization IoU at least 0.50. It is a valuable ranking metric, but redaction still needs one operating threshold and qualitative review.

## <img src="tutorial-icons/claude_task.png" alt="Claude Task" width="36" style="vertical-align:middle; margin-right:9px;"> Claude Coding Task 7: Tune on Validation and Evaluate Test Once

```text
Using plate_model, image_processor, validation_raw, and test_raw:

1. write a batched prediction helper that returns original-size xyxy boxes,
   confidence scores, and ground-truth xyxy boxes;
2. write IoU and one-to-one greedy matching functions at IoU >= .50;
3. on validation_raw compare thresholds .10, .20, .30, .40, .50, and .60;
4. calculate TP, FP, FN, precision, recall, and F2 at each threshold;
5. choose the threshold with the largest validation F2 and name it
   selected_threshold;
6. lock that threshold and evaluate test_raw exactly once;
7. return validation and test dataframes;
8. display six test predictions, including the lowest-confidence true
   detections and any missed plates;
9. never extract or print plate characters.

Return only the Python code.
```

### <img src="tutorial-icons/your_workspace.png" alt="Your Workspace" width="30" style="vertical-align:middle; margin-right:8px;"> Your Workspace

Paste, read, and run Claude's response in the next cell.

### <img src="tutorial-icons/reference_solution.png" alt="Reference Solution" width="30" style="vertical-align:middle; margin-right:8px;"> Reference-Solution Design Notes for Task 7

The full matching implementation is intentionally left for Claude and student review because it is the main Level 1 programming challenge. Verify these invariants:

1. Every ground-truth plate can match at most one prediction.
2. Every prediction can match at most one ground-truth plate.
3. Matching uses IoU at least 0.50.
4. Confidence filtering occurs before counts are calculated.
5. Threshold selection uses validation rows only.
6. The selected threshold is not changed after seeing test results.
7. An image with two plates can contribute two true positives or multiple misses.

A test result should be reported as a result of this dataset, subset, checkpoint, epoch count, IoU rule, and locked threshold—not as universal plate-redaction accuracy.

In [ ]:
def coco_xywh_to_xyxy(boxes):
    converted = []
    for x, y, width, height in boxes:
        converted.append([x, y, x + width, y + height])
    return torch.tensor(converted, dtype=torch.float32).reshape(-1, 4)

def predict_split(split, batch_size=8):
    plate_model.to(device).eval()
    records = []
    for start in range(0, len(split), batch_size):
        stop = min(start + batch_size, len(split))
        rows = [split[index] for index in range(start, stop)]
        images = [row["image"].convert("RGB") for row in rows]
        inputs = image_processor(images=images, return_tensors="pt")
        inputs = {name: tensor.to(device) for name, tensor in inputs.items()}
        with torch.no_grad():
            outputs = plate_model(**inputs)
        target_sizes = torch.tensor(
            [[row["height"], row["width"]] for row in rows],
            device=device,
        )
        predictions = image_processor.post_process_object_detection(
            outputs, threshold=0.0, target_sizes=target_sizes
        )
        for offset, (row, prediction) in enumerate(zip(rows, predictions)):
            keep = prediction["labels"] == 0
            records.append({
                "split_index": start + offset,
                "scores": prediction["scores"][keep].detach().cpu(),
                "boxes": prediction["boxes"][keep].detach().cpu(),
                "targets": coco_xywh_to_xyxy(row["objects"]["bbox"]),
            })
    return records

def box_iou_matrix(predicted, actual):
    if len(predicted) == 0 or len(actual) == 0:
        return torch.zeros((len(predicted), len(actual)))
    top_left = torch.maximum(predicted[:, None, :2], actual[None, :, :2])
    bottom_right = torch.minimum(predicted[:, None, 2:], actual[None, :, 2:])
    intersection_wh = (bottom_right - top_left).clamp(min=0)
    intersection = intersection_wh[..., 0] * intersection_wh[..., 1]
    predicted_area = (
        (predicted[:, 2] - predicted[:, 0]).clamp(min=0)
        * (predicted[:, 3] - predicted[:, 1]).clamp(min=0)
    )
    actual_area = (
        (actual[:, 2] - actual[:, 0]).clamp(min=0)
        * (actual[:, 3] - actual[:, 1]).clamp(min=0)
    )
    union = predicted_area[:, None] + actual_area[None, :] - intersection
    return intersection / union.clamp(min=1e-9)

def match_one_image(record, threshold, iou_threshold=0.50):
    keep = record["scores"] >= threshold
    scores = record["scores"][keep]
    boxes = record["boxes"][keep]
    order = torch.argsort(scores, descending=True)
    boxes = boxes[order]
    ious = box_iou_matrix(boxes, record["targets"])
    unmatched_targets = set(range(len(record["targets"])))
    true_positives = false_positives = 0
    for prediction_index in range(len(boxes)):
        candidates = [
            target_index for target_index in unmatched_targets
            if float(ious[prediction_index, target_index]) >= iou_threshold
        ]
        if not candidates:
            false_positives += 1
            continue
        best_target = max(
            candidates,
            key=lambda target_index: float(ious[prediction_index, target_index]),
        )
        unmatched_targets.remove(best_target)
        true_positives += 1
    return true_positives, false_positives, len(unmatched_targets)

def summarize_threshold(records, threshold):
    tp = fp = fn = 0
    for record in records:
        image_tp, image_fp, image_fn = match_one_image(record, threshold)
        tp += image_tp
        fp += image_fp
        fn += image_fn
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f2 = (5 * precision * recall / (4 * precision + recall)
          if 4 * precision + recall else 0.0)
    return {
        "threshold": threshold, "TP": tp, "FP": fp, "FN": fn,
        "precision": precision, "recall": recall, "F2": f2,
    }

validation_records = predict_split(validation_raw)
candidate_thresholds = [0.10, 0.20, 0.30, 0.40, 0.50, 0.60]
validation_metrics = pd.DataFrame([
    summarize_threshold(validation_records, threshold)
    for threshold in candidate_thresholds
]).sort_values(["F2", "recall", "threshold"], ascending=[False, False, True])
display(validation_metrics)
selected_threshold = float(validation_metrics.iloc[0]["threshold"])
print("Selected validation threshold:", selected_threshold)

# The test split is used once, after the threshold is locked.
test_records = predict_split(test_raw)
test_metrics = pd.DataFrame([
    summarize_threshold(test_records, selected_threshold)
])
display(test_metrics)

# Inspect difficult examples without extracting or printing plate characters.
inspection = []
for record in test_records:
    tp, fp, fn = match_one_image(record, selected_threshold)
    kept_scores = record["scores"][record["scores"] >= selected_threshold]
    inspection.append({
        "split_index": record["split_index"],
        "TP": tp, "FP": fp, "FN": fn,
        "maximum_kept_score": float(kept_scores.max()) if len(kept_scores) else 0.0,
    })
inspection_table = pd.DataFrame(inspection).sort_values(
    ["FN", "maximum_kept_score"], ascending=[False, True]
)
display(inspection_table.head(10))

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for ax, split_index in zip(axes.ravel(), inspection_table.head(6)["split_index"]):
    row = test_raw[int(split_index)]
    record = test_records[int(split_index)]
    ax.imshow(row["image"].convert("RGB"))
    for box in record["targets"]:
        x1, y1, x2, y2 = box.tolist()
        ax.add_patch(Rectangle((x1, y1), x2 - x1, y2 - y1,
                               fill=False, edgecolor="#F4A01C", linewidth=2))
    keep = record["scores"] >= selected_threshold
    for score, box in zip(record["scores"][keep], record["boxes"][keep]):
        x1, y1, x2, y2 = box.tolist()
        ax.add_patch(Rectangle((x1, y1), x2 - x1, y2 - y1,
                               fill=False, edgecolor="#257EA6", linewidth=2))
        ax.text(x1, max(0, y1 - 4), f"{float(score):.2f}", color="#257EA6")
    ax.set_title(f"test index {int(split_index)}")
    ax.axis("off")
plt.tight_layout()
plt.show()

## <img src="tutorial-icons/human_check.png" alt="Human Check" width="30" style="vertical-align:middle; margin-right:8px;"> Human Check

> **Domain expertise required:** The questions below are examples, not a complete checklist. Add a check based on the real application; if you lack that expertise, involve someone who has it.

- **Your own domain question:** What could be wrong here that a generic AI system or checklist would be unlikely to notice?


- Which threshold maximized validation $F_2$?
- How did precision and recall move as the threshold increased?
- How many test plates were missed?
- Were small plates missed more often than large plates?
- Did any predicted box cover only part of the plate?
- Was the threshold changed after test evaluation?
- Would the observed false-negative count be acceptable before public release?

# Part 8: Blur Predicted Plates

Redaction applies a transformation inside each predicted box. Padding expands the box by a fraction of its width and height:

$$
x'_{min}=\max(0,x_{min}-\delta_x),\qquad
x'_{max}=\min(W,x_{max}+\delta_x),
$$

with equivalent $y$ coordinates. Clipping prevents invalid crops. Padding protects plate borders, bolts, and characters near the detector's edge.

Gaussian blur replaces each pixel with a weighted average of nearby pixels. A fixed blur radius has different effects on small and large plates, so the function should use a minimum radius and allow stronger settings.

## <img src="tutorial-icons/claude_task.png" alt="Claude Task" width="36" style="vertical-align:middle; margin-right:9px;"> Claude Coding Task 8: Build the Local Redaction Function

```text
Create redact_license_plates(image, model, processor, threshold,
padding_fraction=.10, minimum_blur_radius=18).

The function must:
1. accept a PIL image or local image path and convert it to RGB;
2. run local model inference with torch.no_grad;
3. post-process predictions to original image size;
4. keep only license_plate predictions at the supplied threshold;
5. pad every box by 10% and clip it to image boundaries;
6. apply Gaussian blur with radius at least 18 to each padded crop;
7. return the redacted image and a metadata dataframe containing only
   score and box coordinates—not plate text;
8. modify no source image on disk;
9. demonstrate it on three test images and save redacted copies only.

Return only the Python code.
```

### <img src="tutorial-icons/your_workspace.png" alt="Your Workspace" width="30" style="vertical-align:middle; margin-right:8px;"> Your Workspace

Paste, read, and run Claude's response in the next cell.

### <img src="tutorial-icons/reference_solution.png" alt="Reference Solution" width="30" style="vertical-align:middle; margin-right:8px;"> Reference Solution for Task 8

In [ ]:
from pathlib import Path
from PIL import Image, ImageFilter

def redact_license_plates(
    image,
    model,
    processor,
    threshold,
    padding_fraction=0.10,
    minimum_blur_radius=18,
):
    original = Image.open(image).convert("RGB") if isinstance(image, (str, Path)) \
        else image.convert("RGB").copy()
    inputs = processor(images=original, return_tensors="pt")
    inputs = {name: tensor.to(device) for name, tensor in inputs.items()}
    model = model.to(device).eval()
    with torch.no_grad():
        outputs = model(**inputs)
    target_sizes = torch.tensor([original.size[::-1]], device=device)
    result = processor.post_process_object_detection(
        outputs, threshold=threshold, target_sizes=target_sizes
    )[0]

    redacted = original.copy()
    records = []
    image_width, image_height = original.size
    for score, label, box in zip(
        result["scores"], result["labels"], result["boxes"]
    ):
        if int(label) != 0:
            continue
        x1, y1, x2, y2 = [float(value) for value in box]
        pad_x = (x2 - x1) * padding_fraction
        pad_y = (y2 - y1) * padding_fraction
        padded = (
            max(0, int(np.floor(x1 - pad_x))),
            max(0, int(np.floor(y1 - pad_y))),
            min(image_width, int(np.ceil(x2 + pad_x))),
            min(image_height, int(np.ceil(y2 + pad_y))),
        )
        crop = redacted.crop(padded)
        radius = max(minimum_blur_radius, int(min(crop.size) * 0.35))
        redacted.paste(crop.filter(ImageFilter.GaussianBlur(radius)), padded)
        records.append({"score": float(score), "box_xyxy": padded})
    return redacted, pd.DataFrame(records)

output_dir = Path("redacted_test_images")
output_dir.mkdir(exist_ok=True)
for display_number, index in enumerate([0, 1, 2], start=1):
    redacted, metadata = redact_license_plates(
        test_raw[index]["image"], plate_model, image_processor,
        selected_threshold,
    )
    destination = output_dir / f"redacted_{display_number}.png"
    redacted.save(destination)
    print(destination, "redactions:", len(metadata))
    display(redacted)

## <img src="tutorial-icons/human_check.png" alt="Human Check" width="30" style="vertical-align:middle; margin-right:8px;"> Redaction Verification Checklist

> **Domain expertise required:** The questions below are examples, not a complete checklist. Add a check based on the real application; if you lack that expertise, involve someone who has it.

- **Your own domain question:** What could be wrong here that a generic AI system or checklist would be unlikely to notice?


- Are all visible plates blurred, including small and partly occluded plates?
- Does padding cover the entire character region and plate border?
- Can any character still be read after zooming?
- Did any box blur an unrelated area?
- Did the function preserve the original file?
- Does metadata exclude plate text and original crops?
- Are failures saved for review without creating a new privacy exposure?

> **Release rule:** If even one visible plate is missed, do not describe the output as anonymized without a human review and correction step.

# AI for Social Good: Protective Vision Systems Can Become Surveillance Systems

The same object-detection technology can serve very different social purposes. Detecting license plates so they can be blurred before publishing street imagery can reduce unnecessary exposure. Detecting, reading, linking, and storing those same plates across time can enable surveillance.

Deployment choices determine whether a technical capability protects people or creates a new risk; privacy and real-world consequences must be evaluated alongside model performance.

Responsible redaction should use:

- **data minimization**—detect the region without performing OCR when OCR is unnecessary;
- local inference when practical;
- deletion or tightly restricted handling of unredacted originals;
- a recall-oriented threshold because a missed plate defeats the protective purpose;
- manual review before public release;
- external testing across countries, weather, camera types, plate sizes, and lighting;
- monitoring for groups or settings with systematically worse detection;
- a correction process when identifying information is missed.

A domain or privacy expert should also ask whether the image needs to be collected or published at all.

> **Social-good principle:** Do not collect or infer identifying information merely because a model can. The protective purpose should determine the minimum data and capability used.


# Tutorial 13 Conclusion

You downloaded a labeled license-plate dataset and a pretrained vision transformer, replaced its COCO detection head, fine-tuned it on plate boxes, selected a validation threshold that weights recall, evaluated the untouched test split, and converted predicted boxes into padded blur regions.

The central technical lesson is that fine-tuning adapts a pretrained representation to a new output definition. The central privacy lesson is that redaction quality is controlled by the entire pipeline—data coverage, box quality, loss, threshold, padding, blur strength, distribution shift, and human review—not by the model alone.

## 🟢 🔎 Pólya Step 4 — Look Back

**Backbone checkpoint.** Do not stop at “it ran.” Ask whether the result answers the original problem, what evidence supports it, what failed, and what should change. Domain expertise matters here because a generic checklist cannot know every real-world failure mode.

**In this tutorial:** Compare validation, test, and external failures and reflect on privacy tradeoffs and limitations.

# <img src="tutorial-icons/look_back.png" alt="Look Back" width="44" style="vertical-align:middle; margin-right:10px;"> Final Reflection

1. Why is YOLOS a transformer but not an LLM?
2. Why is a text-only LLM inappropriate for pixel localization?
3. What is the difference between detection, OCR, and redaction?
4. What does a COCO bounding box contain?
5. What does IoU measure?
6. Why does YOLOS need Hungarian matching?
7. Which pretrained weights were reused and which head was replaced?
8. Why did Level 1 use validation $F_2$ to select a threshold?
9. What does a false negative mean in this privacy task?
10. Why are box padding and blur radius part of the safety design?
11. What does the test result establish and not establish?
12. Why is manual review still required?

# <img src="tutorial-icons/level_2.png" alt="Level 2 Challenge" width="44" style="vertical-align:middle; margin-right:10px;"> Level 2 — Optional Deep Dive

> **Challenge ahead:** Complete Level 1 first. The optional route strengthens training and asks whether the model generalizes beyond the selected dataset.

# Part 9: Train on the Full Supplied Split

Use all 6,176 training images and 1,765 validation images. Compare:

- 1 versus 3 epochs;
- learning rates $2\times10^{-5}$ and $5\times10^{-5}$;
- no augmentation versus mild brightness/contrast and scale augmentation;
- fixed 480-pixel input versus a larger input if GPU memory permits.

Plate boxes are often small. Aggressive random crops can remove the plate or corrupt the box. Any augmentation must transform the image and boxes together.

## <img src="tutorial-icons/theory.png" alt="Theory" width="36" style="vertical-align:middle; margin-right:9px;"> Theory: Small Objects Create a Resolution Tradeoff

If a plate occupies fraction $r$ of the original image width, resizing the whole image to width $W'$ gives an expected plate width near $rW'$. Larger model inputs preserve more plate pixels but increase attention cost. Standard self-attention is approximately $O(N^2)$ in the number of patch tokens, so doubling image width and height can increase token count fourfold and attention interactions roughly sixteenfold.

Select resolution by validation performance across relative box sizes, not by assuming larger is always feasible or better.

## <img src="tutorial-icons/claude_task.png" alt="Claude Task" width="36" style="vertical-align:middle; margin-right:9px;"> Claude Coding Task 9: Run the Full Fine-Tuning Comparison

```text
Using the original supplied train and validation splits:

1. compare the four combinations of epoch count [1, 3] and learning rate
   [2e-5, 5e-5];
2. keep model checkpoint, input size, batch design, seed, and evaluation
   procedure fixed;
3. reinitialize from hustvl/yolos-tiny for every run;
4. select using validation F2 and AP50 only;
5. report GPU time and peak allocated memory;
6. lock the final configuration and evaluate the full 882-image test split once;
7. stratify test recall by relative ground-truth box-area quartile;
8. save the chosen model, processor, threshold, license information,
   and a short model card.

Return only the Python code.
```

# Part 10: External Stress Test With PP4AV

PP4AV contains 3,447 annotated driving images from European cities, including daytime, nighttime, and fisheye views. It was created specifically to benchmark face and plate detection for privacy-preserving autonomous-driving data.

Its full download is about 4.51 GB and the dataset is licensed CC BY-NC-ND 4.0. Read the license and course requirements before downloading or redistributing it. Use it as an **external evaluation set**, not as a silent extension of training.

## <img src="tutorial-icons/theory.png" alt="Theory" width="36" style="vertical-align:middle; margin-right:9px;"> Theory: In-Distribution Accuracy Does Not Measure Distribution Shift

Training and test splits from one dataset can share cameras, sources, plate styles, and annotation decisions. External data may change $P(X)$, $P(Y)$, or $P(Y\mid X)$. Performance can fall even without a code bug.

For redaction, report recall by environment, plate size, lighting, view type, and country when labels permit. A single aggregate score can hide a near-total failure on nighttime or tiny plates.

# Part 11: Safer Video Redaction

Applying the image function independently to video frames can cause flicker: one frame detects a plate and the next misses it. A safer video system should:

1. detect plates on each frame or at a planned interval;
2. associate boxes across nearby frames;
3. smooth coordinates over time;
4. keep a redaction active briefly after a missed detection;
5. blur before writing any public preview;
6. remove audio or metadata only when the release purpose requires it;
7. require a final frame-level review.

A temporal rule trades extra blur for fewer momentary exposures. The acceptable persistence window depends on frame rate, vehicle speed, and the consequences of a visible frame.

# <img src="tutorial-icons/look_back.png" alt="Look Back" width="44" style="vertical-align:middle; margin-right:10px;"> Level 2 Look Back

1. Did additional epochs improve validation recall or only training loss?
2. Which box-size quartile had the lowest test recall?
3. How did external PP4AV performance differ from the selected dataset?
4. Which lighting or view conditions produced the most misses?
5. How did input resolution affect memory and small-plate recall?
6. What temporal rule reduced video flicker?
7. What evidence would be required before describing the tool as reliable anonymization?

# Sources and Course Resources

- Keremberke. [License Plate Object Detection Dataset](https://huggingface.co/datasets/keremberke/license-plate-object-detection). CC BY 4.0.
- HUST Vision Lab. [`hustvl/yolos-tiny`](https://huggingface.co/hustvl/yolos-tiny). Apache 2.0.
- Hugging Face. [Object Detection Task Guide](https://huggingface.co/docs/transformers/tasks/object_detection).
- Hugging Face. [YOLOS Documentation](https://huggingface.co/docs/transformers/model_doc/yolos).
- Fang et al. [You Only Look at One Sequence: Rethinking Transformer in Vision through Object Detection](https://arxiv.org/abs/2106.00666), 2021.
- Trinh et al. [PP4AV: A Benchmarking Dataset for Privacy-Preserving Autonomous Driving](https://openaccess.thecvf.com/content/WACV2023/html/Trinh_PP4AV_A_Benchmarking_Dataset_for_Privacy-Preserving_Autonomous_Driving_WACV_2023_paper.html), WACV 2023.
- Roboflow Universe Projects. [License Plate Recognition Dataset](https://universe.roboflow.com/roboflow-universe-projects/license-plate-recognition-rxg4e). CC BY 4.0.